In [ ]:
%pip -q install "mcp>=1.9,<2" pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 5.0 MB/s eta 0:00:00


In [ ]:
import asyncio, csv, json, re, sys, time, uuid, logging, statistics, zipfile
from pathlib import Path
from datetime import datetime, timezone
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

# ==========================================================
# EXPERIMENT 07: AUTOMATED RECRUITMENT WORKFLOW SYSTEM
# Domain: HR | MCP + Rule-Based Agentic Workflow
# ==========================================================

P = Path("/content/RecruitmentWorkflow")
L = Path("/content/hr_mcp_logs")
P.mkdir(exist_ok=True)
L.mkdir(exist_ok=True)

SERVER = Path("/content/hr_mcp_server.py")
TRACE = L / "mcp_traces.jsonl"
LOG = L / "execution.log"
REPORT = P / "workflow_report.json"
AUDIT = P / "audit_log.csv"
ARCHIVE = Path("/content/Experiment_07_HR_MCP_Artifacts.zip")
TID = "HR-TRACE-" + uuid.uuid4().hex[:8].upper()

for f in (TRACE, LOG):
    f.write_text("", encoding="utf-8")

logging.basicConfig(filename=str(LOG), level=logging.INFO, force=True)

# ----------------------------------------------------------
# 1. Generate 100 fictional candidates and configurable rules
# ----------------------------------------------------------

FIELDS = [
    "candidate_id", "candidate_name", "email", "skills_score",
    "experience_score", "assessment_score", "application_role"
]

records = []
for i in range(1, 101):
    records.append({
        "candidate_id": f"C{i:03d}",
        "candidate_name": f"Candidate {i:03d}",
        "email": f"candidate{i:03d}@example.com",
        "skills_score": str(45 + (i * 7) % 56),
        "experience_score": str(35 + (i * 11) % 66),
        "assessment_score": str(40 + (i * 13) % 61),
        "application_role": "Junior Software Developer"
    })

# Known scores for threshold testing
for idx, score in [(1, 75), (2, 74.99), (3, 50), (4, 49.99)]:
    records[idx]["skills_score"] = str(score)
    records[idx]["experience_score"] = str(score)
    records[idx]["assessment_score"] = str(score)

records[0].update({
    "skills_score": "85", "experience_score": "70",
    "assessment_score": "90"
})  # Expected total score = 82

# Deliberate invalid cases
records[95]["experience_score"] = "-5"       # C096: negative score
records[96]["email"] = ""                     # C097: missing email
records[97]["assessment_score"] = "invalid"  # C098: non-numeric score
records[98]["skills_score"] = "120"          # C099: out of range
records[99]["candidate_id"] = "C003"         # C100: duplicate ID

with (P / "candidates.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader()
    w.writerows(records)

RULES = {
    "rule_version": "1.0",
    "application_role": "Junior Software Developer",
    "criteria": {
        "skills_score": 40,
        "experience_score": 30,
        "assessment_score": 30
    },
    "thresholds": {"shortlist_review": 75, "manual_review": 50}
}
(P / "rules.json").write_text(json.dumps(RULES, indent=2), encoding="utf-8")
(P / "README.md").write_text(
    "HR Recruitment Workflow using MCP\n"
    "Processes synthetic candidate data, validates records, "
    "calculates explainable scores, routes applications, "
    "creates notification drafts and records human reviews.\n",
    encoding="utf-8"
)

# Pytest suite for input, scoring and boundary cases
(P / "test_recruitment.py").write_text('''
import csv, json
from pathlib import Path
ROOT = Path(__file__).parent
def data():
    with (ROOT / "candidates.csv").open() as f:
        return list(csv.DictReader(f))
def rules():
    return json.loads((ROOT / "rules.json").read_text())
def score(k,e,a):
    return round(.4*k + .3*e + .3*a, 2)
def route(s):
    r = rules()["thresholds"]
    if s >= r["shortlist_review"]: return "SHORTLIST_REVIEW"
    if s >= r["manual_review"]: return "MANUAL_REVIEW"
    return "NOT_SHORTLISTED_REVIEW"
def test_100_records(): assert len(data()) == 100
def test_required_columns(): assert set(["candidate_id","email","skills_score"]).issubset(data()[0])
def test_weights_total_100(): assert sum(rules()["criteria"].values()) == 100
def test_score_82(): assert score(85,70,90) == 82
def test_boundary_75(): assert route(75) == "SHORTLIST_REVIEW"
def test_boundary_74_99(): assert route(74.99) == "MANUAL_REVIEW"
def test_boundary_50(): assert route(50) == "MANUAL_REVIEW"
def test_below_50(): assert route(49.99) == "NOT_SHORTLISTED_REVIEW"
def test_missing_email(): assert data()[96]["email"] == ""
def test_invalid_number(): assert data()[97]["assessment_score"] == "invalid"
def test_out_of_range(): assert float(data()[98]["skills_score"]) > 100
def test_negative_score(): assert float(data()[95]["experience_score"]) < 0
def test_duplicate_id(): assert data()[99]["candidate_id"] == data()[2]["candidate_id"]
''', encoding="utf-8")

# ----------------------------------------------------------
# 2. MCP server: HR tools, validation, scoring, review, audit
# ----------------------------------------------------------

SERVER.write_text(r'''
import csv, json, math, re, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from mcp.server.fastmcp import FastMCP

P = Path("/content/RecruitmentWorkflow").resolve()
mcp = FastMCP("HR Recruitment Workflow MCP Server")
FIELDS = ["candidate_id","candidate_name","email","skills_score",
          "experience_score","assessment_score","application_role"]

def out(x):
    return json.dumps(x, ensure_ascii=False)

def safe_path(name):
    p = (P / name).resolve()
    if P not in p.parents:
        raise PermissionError("Path outside project directory")
    return p

@mcp.tool()
def list_files() -> str:
    """List files in the HR recruitment workspace."""
    return out({"status":"success","files":sorted(
        x.name for x in P.iterdir() if x.is_file())})

@mcp.tool()
def load_candidates() -> str:
    """Load applicant records from candidates.csv."""
    try:
        with (P/"candidates.csv").open(newline="",encoding="utf-8") as f:
            reader=csv.DictReader(f)
            if not set(FIELDS).issubset(reader.fieldnames or []):
                return out({"status":"error","message":"Required columns missing"})
            rows=list(reader)
        return out({"status":"success","count":len(rows),"records":rows})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def read_rules() -> str:
    """Read configurable HR scoring weights and review thresholds."""
    try:
        r=json.loads((P/"rules.json").read_text())
        ok=(sum(r["criteria"].values())==100 and
            0 <= r["thresholds"]["manual_review"] <=
            r["thresholds"]["shortlist_review"] <= 100)
        return out({"status":"success" if ok else "error","rules":r})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def inspect_dataset() -> str:
    """Inspect candidate file size and columns."""
    with (P/"candidates.csv").open(newline="",encoding="utf-8") as f:
        reader=csv.DictReader(f); rows=list(reader)
    return out({"status":"success","record_count":len(rows),
                "columns":reader.fieldnames})

@mcp.tool()
def read_file(file_path: str) -> str:
    """Read an allowed text file with project-path protection."""
    try:
        p=safe_path(file_path)
        if not p.is_file():
            return out({"status":"error","error_category":"input_error",
                        "message":"File not found"})
        return out({"status":"success","content":p.read_text(encoding="utf-8")})
    except PermissionError as e:
        return out({"status":"blocked","error_category":"security_error",
                    "message":str(e)})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def evaluate_candidates(candidates_json: str) -> str:
    """Validate, deduplicate, score and route candidate applications."""
    try:
        candidates=json.loads(candidates_json)
        rules=json.loads((P/"rules.json").read_text())
        valid, rejected, seen=[],[],set()
        for raw in candidates:
            c={k:str(raw.get(k,"") or "").strip() for k in FIELDS}
            reasons=[]
            cid=c["candidate_id"]
            if cid and cid in seen:
                reasons.append("Duplicate candidate_id")
            elif cid:
                seen.add(cid)

            for k in FIELDS:
                if not c[k]: reasons.append("Missing field: "+k)

            if c["email"] and not re.fullmatch(
                r"[^@\s]+@[^@\s]+\.[^@\s]+",c["email"]):
                reasons.append("Invalid email format")
            if c["application_role"] and c["application_role"] != rules["application_role"]:
                reasons.append("Role does not match configured role")

            values={}
            for k in ("skills_score","experience_score","assessment_score"):
                try:
                    values[k]=float(c[k])
                    if not math.isfinite(values[k]) or not 0 <= values[k] <= 100:
                        reasons.append(k+" must be between 0 and 100")
                except (ValueError,TypeError):
                    reasons.append("Invalid numeric value: "+k)

            if reasons:
                rejected.append({**c,"validation_status":"INVALID",
                                 "rejection_reason":"; ".join(dict.fromkeys(reasons))})
                continue

            weights=rules["criteria"]
            breakdown={k:{"input_score":values[k],"weight_percent":weights[k],
                "weighted_contribution":round(values[k]*weights[k]/100,2)}
                for k in weights}
            score=round(sum(x["weighted_contribution"] for x in breakdown.values()),2)
            th=rules["thresholds"]
            status=("SHORTLIST_REVIEW" if score>=th["shortlist_review"]
                    else "MANUAL_REVIEW" if score>=th["manual_review"]
                    else "NOT_SHORTLISTED_REVIEW")
            valid.append({"candidate":c,"candidate_id":cid,
                "candidate_name":c["candidate_name"],"email":c["email"],
                "application_role":c["application_role"],
                "validation_status":"VALID","score":score,
                "score_breakdown":breakdown,
                "rule_version":rules["rule_version"],
                "proposed_status":status})
        return out({"status":"success","total_records":len(candidates),
                    "valid_records":valid,"rejected_records":rejected,
                    "duplicate_count":sum("Duplicate candidate_id" in r["rejection_reason"] for r in rejected)})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def generate_notification_drafts(results_json: str) -> str:
    """Generate personalized notification drafts; never send real email."""
    try:
        results=json.loads(results_json); drafts=[]
        for r in results:
            c=r["candidate"]; s=r["proposed_status"]; cid=r["candidate_id"]
            if s=="SHORTLIST_REVIEW":
                subject="Application Update - "+cid
                message=(f"Dear {c['candidate_name']},\\n\\n"
                         f"Your application {cid} is being considered for the next stage. "
                         "This is subject to HR review. The recruitment team will communicate next steps.")
            elif s=="MANUAL_REVIEW":
                subject="Application Under Review - "+cid
                message=(f"Dear {c['candidate_name']},\\n\\n"
                         f"Your application {cid} is under review by the recruitment team. "
                         "We will communicate an update after review.")
            else:
                subject="Application Review Update - "+cid
                message=(f"Dear {c['candidate_name']},\\n\\n"
                         f"Your application {cid} requires further HR review before an outcome is decided. "
                         "This is not a final rejection.")
            drafts.append({"candidate_id":cid,"candidate_name":c["candidate_name"],
                "email":c["email"],"proposed_status":s,"subject":subject,
                "message":message,"notification_status":"DRAFT","sent":False,
                "human_approval_required":True})
        return out({"status":"success","draft_count":len(drafts),"drafts":drafts})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def run_tests() -> str:
    """Run pytest test cases with a 20-second timeout."""
    try:
        r=subprocess.run([sys.executable,"-m","pytest","-q","test_recruitment.py"],
                         cwd=str(P),capture_output=True,text=True,timeout=20)
        text=(r.stdout or "")+"\\n"+(r.stderr or "")
        passed=re.search(r"(\\d+) passed",text)
        failed=re.search(r"(\\d+) failed",text)
        n=int(passed.group(1)) if passed else 0
        bad=int(failed.group(1)) if failed else 0
        return out({"status":"success" if r.returncode==0 else "failed",
                    "return_code":r.returncode,"tests_passed":n,
                    "tests_failed":bad,"tests_total":n+bad,"output":text[-4000:]})
    except subprocess.TimeoutExpired:
        return out({"status":"timeout","error_category":"timeout"})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def record_human_reviews(reviews_json: str, valid_ids_json: str) -> str:
    """Process authorized HR review actions; no notification is dispatched."""
    try:
        reviews=json.loads(reviews_json); ids=set(json.loads(valid_ids_json))
        allowed={"APPROVE","EDIT","HOLD","REQUEST_INFO"}
        editable={"SHORTLIST_REVIEW","MANUAL_REVIEW","NOT_SHORTLISTED_REVIEW","REQUIRES_REVIEW"}
        output=[]
        for r in reviews:
            cid=r.get("candidate_id","")
            role=r.get("reviewer_role","")
            action=r.get("action","")
            if role!="HR_REVIEWER" or not r.get("reviewer_id") or cid not in ids:
                output.append({**r,"review_status":"DENIED",
                    "reviewer_action":"DENIED","final_status":"PENDING_HUMAN_REVIEW",
                    "notification_status":"DRAFT"})
                continue
            if action not in allowed:
                output.append({**r,"review_status":"DENIED",
                    "reviewer_action":"DENIED","final_status":"PENDING_HUMAN_REVIEW",
                    "notification_status":"DRAFT"})
                continue
            status=r.get("proposed_status","REQUIRES_REVIEW")
            ns="DRAFT"
            if action=="APPROVE":
                final=status; review_status="APPROVED"; ns="APPROVED_NOT_SENT"
            elif action=="EDIT":
                final=r.get("edited_status","REQUIRES_REVIEW")
                if final not in editable: final="REQUIRES_REVIEW"
                review_status="EDITED"; ns="EDITED_DRAFT_REQUIRES_REAPPROVAL"
            elif action=="HOLD":
                final="ON_HOLD"; review_status="HELD"
            else:
                final="AWAITING_INFORMATION"; review_status="INFO_REQUESTED"
            output.append({**r,"review_status":review_status,
                "reviewer_action":action,"final_status":final,
                "notification_status":ns,"notification_sent":False})
        return out({"status":"success","reviews":output})
    except Exception as e:
        return out({"status":"error","message":str(e)})

@mcp.tool()
def persist_reports(payload_json: str) -> str:
    """Save CSV outputs, append audit history, and write the workflow report."""
    try:
        d=json.loads(payload_json)
        valid=d["valid_records"]; rejected=d["rejected_records"]
        drafts=d["drafts"]; reviews=d["reviews"]; report=d["report"]
        review_by={r["candidate_id"]:r for r in reviews
                   if r.get("review_status") in
                   {"APPROVED","EDITED","HELD","INFO_REQUESTED"}}
        draft_by={x["candidate_id"]:x for x in drafts}
        result_rows=[]
        for x in valid:
            c=x["candidate"]; cid=x["candidate_id"]
            rv=review_by.get(cid,{})
            b=x["score_breakdown"]
            result_rows.append({
                "candidate_id":cid,"candidate_name":c["candidate_name"],
                "email":c["email"],"application_role":c["application_role"],
                "score":x["score"],
                "skills_contribution":b["skills_score"]["weighted_contribution"],
                "experience_contribution":b["experience_score"]["weighted_contribution"],
                "assessment_contribution":b["assessment_score"]["weighted_contribution"],
                "rule_version":x["rule_version"],"proposed_status":x["proposed_status"],
                "reviewer_action":rv.get("reviewer_action","PENDING"),
                "reviewer_id":rv.get("reviewer_id",""),
                "final_status":rv.get("final_status","PENDING_HUMAN_REVIEW"),
                "notification_status":rv.get("notification_status",
                    draft_by.get(cid,{}).get("notification_status","DRAFT"))
            })
        result_fields=list(result_rows[0]) if result_rows else [
            "candidate_id","candidate_name","email","application_role","score",
            "skills_contribution","experience_contribution","assessment_contribution",
            "rule_version","proposed_status","reviewer_action","reviewer_id",
            "final_status","notification_status"]
        def write_csv(path, rows, fields):
            with path.open("w",newline="",encoding="utf-8") as f:
                w=csv.DictWriter(f,fieldnames=fields,extrasaction="ignore")
                w.writeheader(); w.writerows(rows)
        write_csv(P/"candidate_results.csv",result_rows,result_fields)
        write_csv(P/"rejected_records.csv",rejected,FIELDS+["validation_status","rejection_reason"])
        drafts_out=[]
        for x in drafts:
            y=dict(x)
            if x["candidate_id"] in review_by:
                y["notification_status"]=review_by[x["candidate_id"]]["notification_status"]
            drafts_out.append(y)
        write_csv(P/"notification_drafts.csv",drafts_out,
            ["candidate_id","candidate_name","email","proposed_status","subject",
             "message","notification_status","sent","human_approval_required"])
        write_csv(P/"review_actions.csv",reviews,
            ["candidate_id","reviewer_id","reviewer_role","action","edited_status",
             "notes","review_status","reviewer_action","final_status","notification_status"])
        audit_fields=["timestamp","run_id","candidate_id","validation_status","score",
            "score_breakdown","rule_version","proposed_status","reviewer_action",
            "reviewer_id","final_status","notification_status","error_details"]
        now=datetime.now(timezone.utc).isoformat(); audit_rows=[]
        for x in valid:
            cid=x["candidate_id"]; rv=review_by.get(cid,{})
            audit_rows.append({"timestamp":now,"run_id":report["trace_id"],
                "candidate_id":cid,"validation_status":"VALID","score":x["score"],
                "score_breakdown":json.dumps(x["score_breakdown"]),
                "rule_version":x["rule_version"],"proposed_status":x["proposed_status"],
                "reviewer_action":rv.get("reviewer_action","PENDING"),
                "reviewer_id":rv.get("reviewer_id",""),
                "final_status":rv.get("final_status","PENDING_HUMAN_REVIEW"),
                "notification_status":rv.get("notification_status","DRAFT"),
                "error_details":""})
        for x in rejected:
            audit_rows.append({"timestamp":now,"run_id":report["trace_id"],
                "candidate_id":x.get("candidate_id",""),
                "validation_status":"INVALID","score":"","score_breakdown":"",
                "rule_version":report["rule_version"],"proposed_status":"",
                "reviewer_action":"NOT_APPLICABLE","reviewer_id":"",
                "final_status":"QUARANTINED","notification_status":"NOT_SENT",
                "error_details":x.get("rejection_reason","")})
        existed=(P/"audit_log.csv").exists() and (P/"audit_log.csv").stat().st_size>0
        with (P/"audit_log.csv").open("a",newline="",encoding="utf-8") as f:
            w=csv.DictWriter(f,fieldnames=audit_fields)
            if not existed: w.writeheader()
            w.writerows(audit_rows)
        report["artifacts"]=[str(P/x) for x in
            ["candidate_results.csv","rejected_records.csv","notification_drafts.csv",
             "review_actions.csv","audit_log.csv","workflow_report.json"]]
        report["audit_rows_appended"]=len(audit_rows)
        (P/"workflow_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
        with (P/"audit_log.csv").open(encoding="utf-8") as f:
            audit_count=max(0,sum(1 for _ in f)-1)
        return out({"status":"success","saved_files":report["artifacts"],
                    "audit_rows_appended":len(audit_rows),
                    "audit_history_rows":audit_count})
    except Exception as e:
        return out({"status":"error","message":str(e)})

if __name__=="__main__":
    mcp.run(transport="stdio")
''', encoding="utf-8")

# ----------------------------------------------------------
# 3. Planner, supervisor and MCP client helpers
# ----------------------------------------------------------

ARGS = {
    "list_files":{}, "load_candidates":{}, "read_rules":{},
    "inspect_dataset":{}, "read_file":{"file_path":str},
    "evaluate_candidates":{"candidates_json":str},
    "generate_notification_drafts":{"results_json":str},
    "run_tests":{},
    "record_human_reviews":{"reviews_json":str,"valid_ids_json":str},
    "persist_reports":{"payload_json":str}
}
POLICY = {k:{"risk":"LOW","allowed":True} for k in ARGS}
POLICY.update({
    "evaluate_candidates":{"risk":"MEDIUM","allowed":True},
    "run_tests":{"risk":"MEDIUM","allowed":True},
    "record_human_reviews":{"risk":"MEDIUM","allowed":True},
    "persist_reports":{"risk":"MEDIUM","allowed":True}
})

def planner():
    return [
        {"task_id":"T1","tool":"list_files","arguments":{},"phase":"parallel"},
        {"task_id":"T2","tool":"load_candidates","arguments":{},"phase":"parallel"},
        {"task_id":"T3","tool":"read_rules","arguments":{},"phase":"parallel"},
        {"task_id":"T4","tool":"inspect_dataset","arguments":{},"phase":"parallel"},
        {"task_id":"T5","tool":"evaluate_candidates","arguments":{"candidates_json":"<T2.records>"},"depends_on":["T2","T3"]},
        {"task_id":"T6","tool":"generate_notification_drafts","arguments":{"results_json":"<T5.valid_records>"},"depends_on":["T5"]},
        {"task_id":"T7","tool":"run_tests","arguments":{},"depends_on":["T5"]},
        {"task_id":"T8","tool":"record_human_reviews","arguments":{"reviews_json":"<HR_INPUT>","valid_ids_json":"<T5.valid_ids>"},"depends_on":["T6"]},
        {"task_id":"T9","tool":"persist_reports","arguments":{"payload_json":"<WORKFLOW_REPORT>"},"depends_on":["T6","T7","T8"]}
    ]

def supervise(plan, discovered):
    approved,rejected=[],[]
    for t in plan:
        k,a=t["tool"],t["arguments"]
        reason=None
        if k not in discovered or k not in ARGS: reason="Unknown tool"
        elif set(a)!=set(ARGS[k]): reason="Invalid argument keys"
        elif any(not isinstance(a[x],typ) for x,typ in ARGS[k].items()):
            reason="Invalid argument type"
        else: approved.append(t)
        if reason: rejected.append({"task_id":t["task_id"],"tool":k,"reason":reason})
    return {"approved":approved,"rejected":rejected}

def parse_result(r):
    raw="\n".join(b.text for b in getattr(r,"content",[]) if hasattr(b,"text"))
    try: data=json.loads(raw) if raw else {}
    except Exception: data={"raw_output":raw}
    err=bool(getattr(r,"isError",False) or getattr(r,"is_error",False))
    return data,err

TRACE_ROWS=[]
async def mcp_call(session, tool, args=None, task_id=None):
    start=time.perf_counter(); args=args or {}
    try:
        response=await session.call_tool(tool,arguments=args)
        data,err=parse_result(response)
        status="mcp_error" if err else data.get("status","success")
    except Exception as e:
        data={"status":"error","message":str(e)}; status="error"
    row={"trace_id":TID,"task_id":task_id or tool,"tool":tool,
         "arguments":args,"status":status,
         "execution_time_ms":round((time.perf_counter()-start)*1000,3),
         "timestamp":datetime.now(timezone.utc).isoformat(),
         "output":data}
    TRACE_ROWS.append(row)
    with TRACE.open("a",encoding="utf-8") as f: f.write(json.dumps(row)+"\n")
    logging.info("%s %s %s",tool,status,row["execution_time_ms"])
    return row

# ----------------------------------------------------------
# 4. Complete workflow
# ----------------------------------------------------------

async def run_experiment():
    wall_start=time.perf_counter()
    async with stdio_client(
        StdioServerParameters(command=sys.executable,args=[str(SERVER)]),
        errlog=(L/"server_stderr.log").open("w")
    ) as (rd,wr):
        async with ClientSession(rd,wr) as client:
            await client.initialize()
            advertised=await client.list_tools()
            discovered=[t.name for t in advertised.tools]
            print("="*68)
            print("EXPERIMENT 07: AUTOMATED RECRUITMENT WORKFLOW USING MCP")
            print("="*68)
            print("MCP session: initialized over stdio")
            print("Discovered tools:",discovered)

            plan=planner()
            supervision=supervise(plan,discovered)
            unknown=supervise([{"task_id":"BAD","tool":"delete_candidates","arguments":{}}],discovered)
            print("\nPLANNER: tasks =",len(plan))
            print("SUPERVISOR: approved =",len(supervision["approved"]),
                  "| rejected =",len(supervision["rejected"]))
            print("Unknown-tool test: rejected =",len(unknown["rejected"])==1)
            print("TOOL POLICIES:",json.dumps(POLICY))

            # Phase 1: four independent tools in parallel
            auto_start=time.perf_counter()
            init_tools=["list_files","load_candidates","read_rules","inspect_dataset"]
            initial=await asyncio.gather(*[
                mcp_call(client,k,{},f"INIT-{k}") for k in init_tools
            ])
            init={x["tool"]:x["output"] for x in initial}
            if any(init[k].get("status")!="success" for k in init_tools):
                raise RuntimeError("Ingestion/configuration failed; stopping workflow")
            candidates=init["load_candidates"]["records"]
            rules=init["read_rules"]["rules"]
            print("\nINGESTION:",len(candidates),"records | headers:",init["inspect_dataset"]["columns"])
            print("Parallel phase DOP:",len(init_tools),"| rule version:",rules["rule_version"])

            # Phase 2: validation, deduplication, scoring and routing
            evaluation=(await mcp_call(
                client,"evaluate_candidates",
                {"candidates_json":json.dumps(candidates)},
                "EVALUATE-ALL"
            ))["output"]
            if evaluation.get("status")!="success":
                raise RuntimeError("Candidate evaluation failed")
            valid=evaluation["valid_records"]
            rejected=evaluation["rejected_records"]
            byid={x["candidate_id"]:x for x in valid}
            print("\nVALID:",len(valid),"| QUARANTINED:",len(rejected),
                  "| DUPLICATES:",evaluation["duplicate_count"])
            print("SCORE BREAKDOWN EXAMPLE C001:",
                  json.dumps(byid["C001"]["score_breakdown"]))
            print("ROUTING EXAMPLES:")
            for cid in ["C001","C002","C003","C004","C005"]:
                x=byid[cid]
                print(cid,"score =",x["score"],"=>",x["proposed_status"])

            # Phase 3: drafts and pytest are independent after evaluation
            drafts_task,tests_task=await asyncio.gather(
                mcp_call(client,"generate_notification_drafts",
                    {"results_json":json.dumps(valid)},"GENERATE-DRAFTS"),
                mcp_call(client,"run_tests",{},"PYTEST")
            )
            drafts=drafts_task["output"]["drafts"]
            test_result=tests_task["output"]
            print("\nNOTIFICATION DRAFTS:",len(drafts),
                  "| Actual messages sent: 0")
            print("Sample draft:",json.dumps(drafts[0],indent=2))
            print("PYTEST:",test_result.get("status"),
                  "| passed:",test_result.get("tests_passed"),
                  "| total:",test_result.get("tests_total"))

            # Safety and input-error probes through MCP
            traversal=(await mcp_call(client,"read_file",
                {"file_path":"../../etc/passwd"},"SECURITY-PATH"))["output"]
            missing=(await mcp_call(client,"read_file",
                {"file_path":"missing_file.csv"},"INVALID-FILE"))["output"]
            unauthorized=(await mcp_call(client,"record_human_reviews",{
                "reviews_json":json.dumps([{
                    "candidate_id":"C001","reviewer_id":"TEST",
                    "reviewer_role":"INTERN","action":"APPROVE",
                    "proposed_status":byid["C001"]["proposed_status"]
                }]),
                "valid_ids_json":json.dumps(list(byid))
            },"UNAUTHORIZED-REVIEW"))["output"]
            safety={
                "path traversal blocked":traversal.get("status")=="blocked",
                "missing file handled":missing.get("status")=="error",
                "unauthorized reviewer denied":unauthorized.get("reviews",[{}])[0].get("review_status")=="DENIED",
                "unknown tool rejected by supervisor":len(unknown["rejected"])==1
            }
            print("\nSAFETY TESTS:",json.dumps(safety))

            # Workflow validation
            duplicate_count=sum("Duplicate candidate_id" in x["rejection_reason"] for x in rejected)
            validation={
                "100 input records":len(candidates)==100,
                "95 valid records":len(valid)==95,
                "5 invalid/quarantined":len(rejected)==5,
                "one duplicate detected":duplicate_count==1,
                "C001 score is 82":byid["C001"]["score"]==82,
                "75 routes to shortlist":byid["C001"]["proposed_status"]=="SHORTLIST_REVIEW" and byid["C002"]["score"]==75 and byid["C002"]["proposed_status"]=="SHORTLIST_REVIEW",
                "74.99 routes to manual review":byid["C003"]["score"]==74.99 and byid["C003"]["proposed_status"]=="MANUAL_REVIEW",
                "50 routes to manual review":byid["C004"]["score"]==50 and byid["C004"]["proposed_status"]=="MANUAL_REVIEW",
                "49.99 routes to not-shortlisted review":byid["C005"]["score"]==49.99 and byid["C005"]["proposed_status"]=="NOT_SHORTLISTED_REVIEW",
                "one draft per valid candidate":len(drafts)==len(valid),
                "no automatic dispatch":all(x["sent"] is False for x in drafts),
                "pytest passed":test_result.get("status")=="success",
                "scoring weights sum to 100":sum(rules["criteria"].values())==100
            }
            print("\nVALIDATION CHECKS:")
            for k,v in validation.items(): print(("PASS" if v else "FAIL")+" | "+k)
            print("Passed:",sum(validation.values()),"/",len(validation))

            # Automated processing time excludes reviewer waiting and benchmark
            processing_ms=(time.perf_counter()-auto_start)*1000

            # Phase 4: Human-in-the-loop review for three demonstration candidates
            print("\nHUMAN REVIEW: three selected cases; all other applications remain pending.")
            reviewer_id=input("Reviewer ID [HR-DEMO-01]: ").strip() or "HR-DEMO-01"
            reviewer_role=input("Reviewer role [HR_REVIEWER]: ").strip().upper() or "HR_REVIEWER"
            review_inputs=[]
            for cid in ["C001","C004","C005"]:
                x=byid[cid]
                print(f"\n{cid}: score={x['score']} | proposed={x['proposed_status']}")
                if reviewer_role!="HR_REVIEWER":
                    action="APPROVE"   # Server will deny this role.
                else:
                    action=input("Action APPROVE / EDIT / HOLD / REQUEST_INFO [HOLD]: ").strip().upper() or "HOLD"
                    if action not in {"APPROVE","EDIT","HOLD","REQUEST_INFO"}: action="HOLD"
                edited=""
                if action=="EDIT" and reviewer_role=="HR_REVIEWER":
                    edited=input("Revised status (SHORTLIST_REVIEW / MANUAL_REVIEW / NOT_SHORTLISTED_REVIEW) [MANUAL_REVIEW]: ").strip().upper() or "MANUAL_REVIEW"
                notes=input("Review notes (optional): ").strip() if reviewer_role=="HR_REVIEWER" else "Unauthorized role"
                review_inputs.append({
                    "candidate_id":cid,"reviewer_id":reviewer_id,
                    "reviewer_role":reviewer_role,"action":action,
                    "proposed_status":x["proposed_status"],
                    "edited_status":edited,"notes":notes
                })
            review_result=(await mcp_call(client,"record_human_reviews",{
                "reviews_json":json.dumps(review_inputs),
                "valid_ids_json":json.dumps(list(byid))
            },"HR-REVIEW"))["output"]
            reviews=review_result.get("reviews",[])
            for r in reviews:
                print(r["candidate_id"],"|",r["reviewer_action"],
                      "| final:",r["final_status"],"| notification:",r["notification_status"])

            # Phase 5: Sequential versus parallel MCP benchmark
            benchmark=["list_files","load_candidates","read_rules","inspect_dataset"]
            seq,par=[],[]
            for run in range(1,6):
                st=time.perf_counter()
                for j,k in enumerate(benchmark):
                    await mcp_call(client,k,{},f"SEQ-{run}-{j+1}")
                seq.append((time.perf_counter()-st)*1000)
                st=time.perf_counter()
                await asyncio.gather(*[
                    mcp_call(client,k,{},f"PAR-{run}-{j+1}")
                    for j,k in enumerate(benchmark)
                ])
                par.append((time.perf_counter()-st)*1000)
            avg_s=statistics.mean(seq); avg_p=statistics.mean(par)
            reduction=(avg_s-avg_p)/avg_s*100 if avg_s else 0
            print("\nSEQUENTIAL vs PARALLEL")
            for i,(a,b) in enumerate(zip(seq,par),1):
                print(i,round(a,3),"ms |",round(b,3),"ms | saved",round(a-b,3),"ms")
            print("Average sequential:",round(avg_s,3),"ms")
            print("Average parallel:",round(avg_p,3),"ms")
            print("Time reduction:",round(reduction,2),"%")

            # Metrics and routing summary
            route_counts={s:sum(1 for x in valid if x["proposed_status"]==s)
                for s in ["SHORTLIST_REVIEW","MANUAL_REVIEW","NOT_SHORTLISTED_REVIEW"]}
            reviewed=sum(r.get("review_status") in {"APPROVED","EDITED","HELD","INFO_REQUESTED"} for r in reviews)
            total_wall_ms=(time.perf_counter()-wall_start)*1000
            test_total=test_result.get("tests_total",0)
            test_pass=test_result.get("tests_passed",0)
            metrics={
                "total_candidate_records":len(candidates),
                "valid_records":len(valid),
                "invalid_or_quarantined_records":len(rejected),
                "duplicate_records_detected":duplicate_count,
                "shortlist_review":route_counts["SHORTLIST_REVIEW"],
                "manual_review":route_counts["MANUAL_REVIEW"],
                "not_shortlisted_review":route_counts["NOT_SHORTLISTED_REVIEW"],
                "notification_drafts_generated":len(drafts),
                "human_review_actions_completed":reviewed,
                "human_review_completion_rate_for_three_demo_cases_percent":round(reviewed/3*100,2),
                "review_completion_rate_for_all_valid_applications_percent":round(reviewed/len(valid)*100,2),
                "notifications_sent":0,
                "notification_failures":0,
                "data_validation_rate_percent":round(len(valid)/len(candidates)*100,2),
                "workflow_processing_success_rate_percent":round(len(valid)/len(valid)*100,2) if valid else 0,
                "notification_draft_generation_rate_percent":round(len(drafts)/len(valid)*100,2) if valid else 0,
                "test_cases_passed":test_pass,
                "test_cases_total":test_total,
                "test_pass_rate_percent":round(test_pass/test_total*100,2) if test_total else 0,
                "automated_processing_time_ms":round(processing_ms,3),
                "average_processing_ms_per_valid_record":round(processing_ms/len(valid),3) if valid else 0,
                "elapsed_before_report_save_ms":round(total_wall_ms,3),
                "sequential_times_ms":[round(x,3) for x in seq],
                "parallel_times_ms":[round(x,3) for x in par],
                "average_sequential_ms":round(avg_s,3),
                "average_parallel_ms":round(avg_p,3),
                "parallel_time_reduction_percent":round(reduction,2),
                "benchmark_DOP":4,
                "main_phase_DOP":4
            }

            final_status="VALIDATED" if (
                all(validation.values()) and all(safety.values())
                and supervision["rejected"]==[]
            ) else "REQUIRES_REVIEW"

            report={
                "experiment":"Experiment 07",
                "title":"Automated Recruitment Workflow System using MCP",
                "domain":"HR",
                "use_case":"Junior Software Developer recruitment",
                "trace_id":TID,"rule_version":rules["rule_version"],
                "mcp_tools_discovered":discovered,
                "planner_tasks":plan,
                "supervisor":supervision,
                "validation":validation,"safety_checks":safety,
                "test_result":test_result,"metrics":metrics,
                "workflow_status":final_status,
                "notification_policy":"Drafts only; no real emails sent.",
                "human_review_policy":"Only HR_REVIEWER role accepted by the demo policy.",
                "audit_policy":"Audit CSV appends rows to preserve history."
            }
            payload={
                "valid_records":valid,"rejected_records":rejected,
                "drafts":drafts,"reviews":reviews,"report":report
            }
            saved=(await mcp_call(client,"persist_reports",{
                "payload_json":json.dumps(payload)
            },"PERSIST-REPORTS"))["output"]
            print("\nPERSISTENCE:",json.dumps(saved,indent=2))

            # Bundle all files, including the final MCP trace
            with zipfile.ZipFile(ARCHIVE,"w",zipfile.ZIP_DEFLATED) as z:
                for f in P.iterdir():
                    if f.is_file(): z.write(f,f"RecruitmentWorkflow/{f.name}")
                for f in [TRACE,LOG,L/"server_stderr.log",SERVER]:
                    if f.exists(): z.write(f,f"logs/{f.name}")

            print("\nFINAL EXPERIMENT SUMMARY")
            print(json.dumps({
                "final_status":final_status,"trace_id":TID,
                "mcp_tools":len(discovered),
                "planned_tasks":len(plan),
                "approved_tasks":len(supervision["approved"]),
                "valid_candidates":len(valid),
                "quarantined_candidates":len(rejected),
                "validation_checks_passed":sum(validation.values()),
                "safety_checks_passed":sum(safety.values()),
                "metrics":metrics,
                "artifact_folder":str(P),
                "artifact_zip":str(ARCHIVE)
            },indent=2))
            print("\nFILES:")
            for f in sorted(P.iterdir()):
                if f.is_file(): print(f.name,"|",f.stat().st_size,"bytes")
            print("\nZIP:",ARCHIVE)
            return report

FINAL_REPORT = await run_experiment()

EXPERIMENT 07: AUTOMATED RECRUITMENT WORKFLOW USING MCP
MCP session: initialized over stdio
Discovered tools: ['list_files', 'load_candidates', 'read_rules', 'inspect_dataset', 'read_file', 'evaluate_candidates', 'generate_notification_drafts', 'run_tests', 'record_human_reviews', 'persist_reports']

PLANNER: tasks = 9
SUPERVISOR: approved = 9 | rejected = 0
Unknown-tool test: rejected = True
TOOL POLICIES: {"list_files": {"risk": "LOW", "allowed": true}, "load_candidates": {"risk": "LOW", "allowed": true}, "read_rules": {"risk": "LOW", "allowed": true}, "inspect_dataset": {"risk": "LOW", "allowed": true}, "read_file": {"risk": "LOW", "allowed": true}, "evaluate_candidates": {"risk": "MEDIUM", "allowed": true}, "generate_notification_drafts": {"risk": "LOW", "allowed": true}, "run_tests": {"risk": "MEDIUM", "allowed": true}, "record_human_reviews": {"risk": "MEDIUM", "allowed": true}, "persist_reports": {"risk": "MEDIUM", "allowed": true}}

INGESTION: 100 records | headers: ['candidate